In [85]:
from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [86]:
from xgboost import XGBClassifier

In [87]:
%matplotlib inline

In [88]:
data_path = Path('data', 'heart_failure_prediction.csv')
df = pd.read_csv(data_path)
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [89]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

In [90]:
num_attributes = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
cat_attributes = ['Sex', 'ChestPainType', 'FastingBS', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

In [91]:
predicted_feature = 'HeartDisease'

X = df.drop(predicted_feature, axis=1)
y = df[predicted_feature]

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=True, train_size=0.8)

In [92]:
# Check unique values in the categorical columns
for col in cat_attributes:
    print(f"Unique values in {col}: {X_train[col].unique()}")

Unique values in Sex: ['F' 'M']
Unique values in ChestPainType: ['ASY' 'NAP' 'ATA' 'TA']
Unique values in FastingBS: [0 1]
Unique values in RestingECG: ['LVH' 'ST' 'Normal']
Unique values in ExerciseAngina: ['N' 'Y']
Unique values in ST_Slope: ['Up' 'Down' 'Flat']


In [93]:
categories = [df[cat_attribute].unique() for cat_attribute in cat_attributes]

col_transformer = make_column_transformer(
    (StandardScaler(), num_attributes),
    (OrdinalEncoder(categories=categories), cat_attributes)
)

In [94]:
# Apply transformations to the training data
X_train_transformed = pd.DataFrame(col_transformer.fit_transform(X_train), columns=num_attributes + cat_attributes)

# Display the first few rows of the transformed data
print(X_train_transformed.head())

        Age  RestingBP  Cholesterol     MaxHR   Oldpeak  Sex  ChestPainType  \
0 -0.086209   0.287755     0.331233  0.924141 -0.834569  1.0            2.0   
1  2.143372   0.935645     0.554317 -0.245510  2.981495  0.0            2.0   
2  1.294008   1.205599    -1.843839 -1.805045 -0.834569  1.0            2.0   
3 -1.254085  -0.954034     0.117443  0.027409 -0.834569  1.0            1.0   
4 -0.086209   0.395737     0.043082  0.729199  2.122880  0.0            2.0   

   FastingBS  RestingECG  ExerciseAngina  ST_Slope  
0        0.0         2.0             0.0       0.0  
1        1.0         1.0             1.0       2.0  
2        1.0         0.0             0.0       1.0  
3        0.0         1.0             0.0       0.0  
4        1.0         2.0             1.0       2.0  


In [95]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [96]:
X_train_transformed.columns

Index(['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak', 'Sex',
       'ChestPainType', 'FastingBS', 'RestingECG', 'ExerciseAngina',
       'ST_Slope'],
      dtype='object')

In [97]:
logisitc_regressor = LogisticRegression()
model = logisitc_regressor.fit(X_train_transformed, y_train)

In [98]:
np.mean(cross_val_score(logisitc_regressor, X_train_transformed, y_train))

np.float64(0.8515981735159818)

In [99]:
np.mean(cross_val_score(KNeighborsClassifier(), X_train_transformed, y_train))

np.float64(0.8461094026651758)

In [100]:
np.mean(cross_val_score(SVC(), X_train_transformed, y_train))

np.float64(0.8665455223185164)

In [130]:
param_grid = {
    'C': np.arange(0.1, 1.0, 0.1),
    'kernel': ['poly'],
    'degree': range(1, 10),
    'gamma': range(1, 10)
}

grid = GridSearchCV(SVC(), param_grid, refit=True, verbose=1, n_jobs=8)
model = grid.fit(X_train_transformed, y_train)

Fitting 5 folds for each of 729 candidates, totalling 3645 fits


In [131]:
model.best_params_

{'C': np.float64(0.9), 'degree': 1, 'gamma': 8, 'kernel': 'poly'}

In [132]:
X_test_transformed = pd.DataFrame(col_transformer.transform(X_test), columns=X_train_transformed.columns)

In [133]:
predictions = model.predict(X_test_transformed)

In [134]:
y_test

573    1
310    0
248    1
869    0
834    0
      ..
16     1
324    1
586    1
861    1
457    1
Name: HeartDisease, Length: 184, dtype: int64

In [135]:
accuracy_score(y_test, predictions)

0.8260869565217391

In [119]:
param_grid = {
    'C': np.arange(0.1, 1.0, 0.1),
    'kernel': ['linear', 'rbf', 'sigmoid', 'poly',],
    'gamma': range(1, 5)
}

grid = GridSearchCV(SVC(), param_grid, refit=True, verbose=1)
model = grid.fit(X_train_transformed, y_train)

Fitting 5 folds for each of 144 candidates, totalling 720 fits


In [120]:
model.best_params_

{'C': np.float64(0.5), 'gamma': 1, 'kernel': 'linear'}

In [121]:
predictions = model.predict(X_test_transformed)
accuracy_score(y_test, predictions)

0.8260869565217391